In [4]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-5-5"

In [5]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)

    for block in message.content:
        if block.type == "text":
            return block.text

## 課題: 評価パイプラインの骨組み(3つの関数)を作る

- 対応: L12「評価の実行」。L10 の5ステップのうち、ステップ3(Claude に通す)と、ステップ4(採点者に通す)の枠を作る。
- 今回、新しく覚えること: 3つの関数(`run_prompt` → `run_test_case` → `run_eval`)を組み合わせて、データセットの全件を処理する。
- 目標: ex07 で作った `dataset.json` を読み込み、全件を Claude に通して、`output`・`test_case`・`score` を持つ結果のリストを得る。
- 採点は、まだ作らない。`score` は、仮の値(10)にしておく(L12 の確認問題で答えた、パイプライン全体を動かすための置き場所)。
- プロンプトは、わざと最小にする(ベースライン)。形式の指示は入れない。
- `dataset.json` は、ex07 のフォルダにある。コピーせず、相対パスで読む(`../ex07_Generating-test-datasets/dataset.json`)。うまく読めないときは、ヒントを見る。
- `chat()` は、これまでのもの(`system` 引数のみ)を、そのまま使う。
- API を3回呼ぶので、少額の料金がかかる。実行には、30秒ほどかかることがある。
- ヒントは、下の「ヒント」のセルにある。まず、見ずに、試してみる。

In [6]:
import json

# TODO 1: dataset.json を読み込んで、dataset に入れる
#   - ex07 では、保存するとき encoding="utf-8" を付けた。読むときも、同じ指定が要るかを考える。
#   - ページのコードは encoding を付けていない。付けない場合に何が起きるかも、試してみてよい。
with open("../ex07_Generating-test-datasets/dataset.json", "r", encoding="utf-8") as f:  # TODO: encoding の指定
    dataset = json.load(f)

print(len(dataset))
print(dataset[0])

3
{'task': 'S3バケット名が命名規則（小文字・数字・ハイフンのみ、3〜63文字、先頭と末尾は英数字）を満たしているかを判定する単一のPython関数 is_valid_s3_bucket_name(name: str) -> bool を書いてください。'}


In [7]:
# TODO 2: run_prompt を完成させる
#   - test_case["task"] を、プロンプトに差し込む。
#   - messages に、ユーザーメッセージとして追加して、chat() を呼ぶ。
#   - Claude の出力(文字列)を返す。
def run_prompt(test_case):
    """プロンプトとテストケースを結合して、Claude の出力を返す"""
    prompt = f"""
次のタスクを解決してください:

{test_case["task"]}
"""  # TODO: 差し込む値
    messages = []
    # TODO: add_user_message で prompt を追加する
    add_user_message(messages, prompt)
    output = chat(messages)  # TODO: chat(messages) を呼ぶ
    return output

In [8]:
# TODO 3: run_test_case を完成させる
#   - run_prompt を呼んで、出力を得る。
#   - 採点は、まだ作らない。score は、仮の値 10 にしておく。
#   - "output"、"test_case"、"score" の3つを持つ辞書を返す。
def run_test_case(test_case):
    """run_prompt を呼び、結果を採点する(採点は仮の値)"""
    output = run_prompt(test_case)  # TODO: run_prompt を呼ぶ
    score = 10  # 仮の値(後で、本物の採点に置き換える)
    return {
        "output": output,
        "test_case": test_case,  # TODO
        "score": score,
    }

In [9]:
# TODO 4: run_eval を完成させる
#   - dataset の各ケースを、順に run_test_case に渡す。
#   - 結果を、results のリストに追加していく。
def run_eval(dataset):
    """データセットの全ケースを処理して、結果のリストを返す"""
    results = []
    # TODO: for で dataset を回し、run_test_case の結果を results に追加する

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results

In [10]:
# TODO 5: 評価を実行して、結果を表示する
#   - 時間がかかるので、実行中は待つ。
#   - 結果は、json.dumps で整形して表示する。日本語をそのまま見たいので ensure_ascii=False を付ける。

results = run_eval(dataset)
print(json.dumps(results, indent=2, ensure_ascii=False))

[
  {
    "output": "```python\nimport re\n\n_S3_BUCKET_NAME_PATTERN = re.compile(r\"[a-z0-9][a-z0-9-]{1,61}[a-z0-9]\")\n\n\ndef is_valid_s3_bucket_name(name: str) -> bool:\n    \"\"\"\n    S3バケット名が次の規則を満たすかを判定する。\n    - 長さが3〜63文字\n    - 使用できる文字は小文字英字・数字・ハイフンのみ\n    - 先頭と末尾は英小文字または数字\n    \"\"\"\n    # re.fullmatch を使うことで、末尾の改行などによる誤判定を防ぐ\n    return _S3_BUCKET_NAME_PATTERN.fullmatch(name) is not None\n\n\nif __name__ == \"__main__\":\n    tests = {\n        \"my-bucket\": True,\n        \"abc\": True,\n        \"a1-b2-c3\": True,\n        \"ab\": False,                 # 2文字（短すぎる）\n        \"a\" * 63: True,              # 63文字\n        \"a\" * 64: False,             # 64文字（長すぎる）\n        \"-bucket\": False,            # 先頭がハイフン\n        \"bucket-\": False,            # 末尾がハイフン\n        \"My-Bucket\": False,          # 大文字を含む\n        \"my_bucket\": False,          # アンダースコアを含む\n        \"my.bucket\": False,          # ピリオドは本仕様では不可\n        \"バ",
    "test_case": {
      "task": "S3バケッ

## ヒント(困ったときだけ開く)

**TODO 1(読み込み)**
- 書き込みのときに付けた指定と、同じものを付ける(`encoding="utf-8"`)。
- `FileNotFoundError` が出たら、ノートブックがあるフォルダが、どこになっているかを考える。相対パスは、そこから数える。
- 読み込みに失敗したら、エラーの最後の行を読む(`UnicodeDecodeError` なら、文字コードの問題)。

**TODO 2(`run_prompt`)**
- f文字列(先頭に `f` が付いた文字列)の中で、`{}` に入れた式が、値に置き換わる。
- 入れる値は、`test_case` の `"task"`(辞書からキーで取り出す書き方は、L11 の `data["tasks"]` と同じ)。
- `add_user_message(messages, prompt)` と `chat(messages)` は、これまでと同じ使い方。

**TODO 3(`run_test_case`)**
- `run_prompt(test_case)` の戻り値を、`output` に入れる。
- 辞書の `"test_case"` には、引数で受け取った `test_case` を、そのまま入れる。

**TODO 4(`run_eval`)**
- L07 や L04 で見た `for` の形: `for test_case in dataset:`
- リストへの追加は、`results.append(...)`。

**TODO 5(実行)**
- ここは書き換えなくてよい。エラーが出たら、どの関数の中で起きたかを、エラーの `--->` の行から探す。

**エラーが出たら**
- 最後の行を読む。API のエラーは、400 などの番号と `message` に、どの項目が悪いかが書かれていることが多い。
- 途中の関数が `None` を返していると、後ろの処理で `NoneType` のエラーになる。

## 振り返り(自分の言葉で書く)

1. 結果の `output` を読んで、プロンプトを最小にしたことで、どんな出力になったか。形式の指示がないと、何が起きるか。
2. `run_prompt`・`run_test_case`・`run_eval` の3つに分けるメリットは何か。1つの関数にまとめた場合と比べて、あとで採点を差し替えるとき、どこを直せばよいか。
3. `dataset.json` の読み込みで、`encoding` を指定しない場合に、実際に何が起きたか(試した場合)。
4. 3件の処理にかかった時間から、データセットが100件、1000件になったら、どうなりそうか。